# Vector Search via MCP

This is the first of three notebooks that explore **graph-enriched search** — retrieval strategies that take advantage of the knowledge graph's structure. This notebook starts with the foundation: **semantic vector search**.

**Quick background:** The SEC 10-K filings in this workshop were split into smaller passages (chunks), and each chunk was encoded as a numerical vector (embedding) that captures its meaning. Vector search finds chunks whose meaning is closest to your question. Graph-enriched search then follows relationships from those chunks to connected entities like companies, products, and risk factors. This notebook covers vector search; the next two notebooks add graph traversal and hybrid search.

The agent communicates with Neo4j through MCP (Model Context Protocol), an open standard for agent-to-tool communication. MCP is the transport layer — the interesting part is what you retrieve and how.

**Learning Objectives:**
- Perform vector similarity search against a Neo4j knowledge graph
- Encapsulate embedding generation inside a custom `@tool` wrapper so the LLM never sees raw float arrays
- Understand what vector search returns (isolated chunks) and what it misses (graph context)
- Connect to Neo4j through an MCP server via AgentCore Gateway

**How it works:**
1. A custom `@tool` wrapper encapsulates embedding generation and Cypher query execution
2. The agent calls `vector_search_tool("query text")` — the embedding stays on the data plane, never passing through the LLM's context window
3. Neo4j's vector index returns the most semantically similar chunks
4. The agent formats and presents the results

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database, embeddings, and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx nest-asyncio -q

## 1. Configuration and Imports

In [ ]:
import os

import nest_asyncio
from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent, tool
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

nest_asyncio.apply()

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("\nConfiguration loaded!")

## 2. Shared Utilities

This notebook imports the embedding helper from `lib/` rather than defining it inline:

- **`lib/lab_4_data_utils.py`** centralizes Bedrock embedding configuration — model ID, dimensions, and client lifecycle — so every notebook uses consistent settings. Changing the embedding model or dimensions in one place updates all labs.

In [ ]:
from lib.lab_4_data_utils import get_embedding

# Verify the shared embedding function works
test_embedding = get_embedding("What are Apple's main products?")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

## 3. Initialize Model and MCP Connection

- **BedrockModel**: Configures Claude on Amazon Bedrock with temperature 0 for deterministic responses
- **MCPClient**: Strands' MCP integration — discovers available tools and provides `call_tool_sync()` for direct tool invocation inside `@tool` wrappers

In [ ]:
from botocore.config import Config as BotocoreConfig

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools and find the Cypher query tool
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")

cypher_tool = next((n for n in tool_names if "read-cypher" in n), None)
assert cypher_tool, f"No Cypher query tool found among: {tool_names}"

print(f"\nModel: {MODEL_ID}")
print(f"Cypher tool: {cypher_tool}")

## 4. Vector Search Tool and Agent

The `@tool` wrapper is the key pattern: embedding generation and Cypher execution are encapsulated inside a custom tool function. The agent calls `vector_search_tool("Apple products")` — a clean string-in, string-out interface that keeps the 1024-dimensional embedding on the data plane, never passing it through the LLM's context window.

This avoids wasting tokens on numeric arrays the LLM cannot interpret and eliminates the risk of the LLM truncating or modifying the embedding.

In [ ]:
@tool
def vector_search_tool(query: str, top_k: int = 5) -> str:
    """Search for semantically similar chunks using vector embeddings.
    Use this for semantic queries about SEC 10-K filing data."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        WITH node {{.*, embedding: null}} AS node, score
        RETURN node.text AS text, score
        ORDER BY score DESC
    """

    result = mcp_client.call_tool_sync(
        tool_use_id="vector-search",
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


VECTOR_SEARCH_PROMPT = """You are a retrieval assistant that performs semantic vector search against a Neo4j database containing SEC 10-K filing data.

You have a vector_search_tool that finds semantically similar text chunks. Call it with a natural language query and an optional top_k parameter.

FORMAT:
For each result, show:
1. The similarity score (higher = more similar)
2. A preview of the chunk text (first 200 characters)
"""

agent = Agent(
    model=bedrock_model,
    system_prompt=VECTOR_SEARCH_PROMPT,
    tools=[vector_search_tool],
)


async def vector_search(query: str, top_k: int = 5):
    """Run vector search through the agent."""
    print(f'Query: "{query}"')
    print(f"Top K: {top_k}")
    print("-" * 60)

    response = await agent.invoke_async(
        f"Search for: {query}\nUse top_k={top_k}."
    )
    print(f"\n{response}")
    return response


print("Vector search agent ready!")

## 5. Run Vector Searches

Search for semantically similar chunks. The vector index finds chunks whose embeddings are closest to the query embedding, even when the exact words differ from the source text.

In [ ]:
result = await vector_search("What are Apple's main products?", top_k=5)

In [ ]:
result = await vector_search("What are the key risk factors mentioned in SEC filings?", top_k=5)

In [ ]:
# Compare top_k values — fewer results for a focused search
result = await vector_search("What financial metrics indicate company performance?", top_k=3)

## Summary

You performed semantic vector search against a Neo4j knowledge graph:

| Component | Role |
|-----------|------|
| **`@tool` wrapper** | Encapsulates embedding generation + Cypher execution — the LLM never sees the float array |
| **Bedrock Nova** | Embeds the query text into a 1024-dimensional vector (inside the tool) |
| **Strands Agent** | Decides what to search for and calls `vector_search_tool` with natural language |
| **MCPClient** | Executes the Cypher query against Neo4j via `call_tool_sync()` |
| **Neo4j Vector Index** | Returns chunks ranked by cosine similarity |

The vector search returns isolated chunks — the most semantically similar text fragments. It doesn't know about the chunk's document, its neighbors, or related entities in the graph.

In the next notebook, we add **graph traversal** to pull in surrounding context alongside each vector match. This is what makes retrieval from a knowledge graph different from a standalone vector database.

---

**Next:** [Graph-Enriched Search](02_graph_enriched_search_mcp.ipynb)